In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [ ]:
# Load the preprocessed data
data = pd.read_csv('feature_matrix.csv')

# Explore the data
print(data.head())
print(data.info())

In [ ]:
data = data.replace(-np.inf, np.nan)
data = pd.get_dummies(data)

# Compute column-wise maximums, ignoring NaNs

column_max = data.max(axis=0, skipna=True)
 
# Replace NaNs (former infs) with the negative maximum of the column

for col in data.columns:
    if data[col].dtype != bool:
        data[col] = data[col].fillna(-column_max[col])
    else:
        data[col] = data[col].fillna(False)
 
#remove columns containing only nan
data = data.dropna(axis=1, how='all')

# Identify columns with NaNs
nan_columns = data.columns[data.isnull().any()].tolist()

print(f"Columns with NaNs: {nan_columns}")

# Check for any remaining NaNs or infinities
if data.isnull().values.any():
    raise ValueError("Data contains NaNs or infinities after preprocessing")

# Prepare features and target
X = data.drop('TARGET', axis=1).values
y = data['TARGET'].values

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert data to PyTorch tensors
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

# Define the neural network model
class RiskModel(nn.Module):
    def __init__(self):
        super(RiskModel, self).__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)  # Binary classification

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = RiskModel()

# Specify loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
epochs = 50
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    # ...existing code...
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}')

# Evaluate the model
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / y_test.size(0)
    print(f'Accuracy on test data: {accuracy * 100:.2f}%')

In [ ]:
#show roc curve and f1 score
from sklearn.metrics import roc_curve, f1_score
import matplotlib
import matplotlib

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import f1_score

# Compute predicted probabilities
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    probs = torch.nn.functional.softmax(outputs, dim=1)[:, 1].numpy()

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test, probs)

# Compute AUC
auc = roc_auc_score(y_test, probs)

# Compute F1 score
threshold = thresholds[np.argmax(2 * tpr * (1 - fpr))]
y_pred = probs > threshold
f1 = f1_score(y_test, y_pred)

# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()

print(f'F1 score: {f1:.2f}')

#show confusion matrix
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()
